In [0]:
# O QUE: Ler os pedidos originais do Olist usando PySpark.
# POR QUÊ: Validar a fonte antes de construir a tabela Bronze.

caminho_csv = "/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv"

df_orders_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(caminho_csv)
)

# Validar quantidade de registros
print(f"Total de registros: {df_orders_raw.count()}")

# Inspecionar o schema
df_orders_raw.printSchema()

# Visualizar cinco registros
display(df_orders_raw.limit(5))


In [0]:

# O QUE: Criar a Bronze somente se a tabela não existir.
# POR QUÊ: Evitar sobrescrever dados em novas execuções.

tabela = "workspace.bronze.orders"

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

if not spark.catalog.tableExists(tabela):
    (
        df_orders_raw.write
        .format("delta")
        .saveAsTable(tabela)
    )
    print("Tabela Bronze criada.")
else:
    print("Tabela Bronze já existe. Nenhuma gravação realizada.")

# Validar os dados existentes
print(f"Total de registros: {spark.table(tabela).count()}")

display(
    spark.sql(f"DESCRIBE DETAIL {tabela}")
    .select("format", "numFiles", "sizeInBytes")
)
